# Assignment 1: Text Chunking, Embeddings, and Semantic Search

**Course:** Developing Generative AI Solutions

**Student:** write your name here

This notebook builds the retrieval half of a RAG system on one text file:

1. Load a plain text file that has 12 paragraphs.
2. Split it into fixed-size chunks of 500 characters with a 100-character overlap (20%).
3. Convert every chunk into a vector with an embedding model (`gemini-embedding-001`).
4. Take a search query from the user.
5. Embed the query with the same model.
6. Compute the cosine similarity between the query vector and every chunk vector.
7. Rank the chunks and print the question with the top 3 chunks and their scores.

**To run:** add a Gemini API key to Colab Secrets under the name `GEMINI_API_KEY` and turn on notebook access for it. Then run the cells from top to bottom.

## Setup

Install the Gemini SDK, import the libraries, and create the API client. The key is read from Colab Secrets, so it never appears in the notebook.

In [ ]:
!pip install -q google-genai

In [ ]:
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
EMBED_MODEL = "gemini-embedding-001"

## Data file

This cell creates `data.txt` inside Colab. It is an Arabic text about 12 Saudi cities and governorates, one paragraph each. To use another file, upload it to Colab and change the file name in step 1.

In [ ]:
%%writefile data.txt
الرياض عاصمة المملكة العربية السعودية وأكبر مدنها من حيث عدد السكان، وتقع في وسط المملكة على هضبة نجد. من أشهر معالمها التاريخية قصر المصمك الذي استعاده الملك عبدالعزيز عام 1902، وبالقرب منها تقع الدرعية عاصمة الدولة السعودية الأولى، وفيها حي الطريف المسجل في قائمة التراث العالمي لليونسكو منذ عام 2010. ومن معالمها الحديثة برج المملكة وبرج الفيصلية ومركز الملك عبدالله المالي. افتُتح قطار الرياض في أواخر عام 2024، وهو شبكة مترو من ستة مسارات. وقد فازت الرياض باستضافة معرض إكسبو 2030.

مكة المكرمة أقدس مدينة في الإسلام، وتقع في غرب المملكة. فيها المسجد الحرام الذي تتوسطه الكعبة المشرفة، قبلة المسلمين في صلاتهم، ويقصدها ملايين المسلمين كل عام لأداء الحج والعمرة. وُلد فيها النبي محمد صلى الله عليه وسلم، وفيها جبل النور الذي يضم غار حراء حيث نزل الوحي أول مرة. بالقرب منها المشاعر المقدسة: منى وعرفات ومزدلفة. ومن أبرز مبانيها الحديثة برج الساعة المطل على المسجد الحرام، وهو من أطول المباني في العالم.

المدينة المنورة ثاني أقدس مدينة في الإسلام، وتقع في غرب المملكة شمال مكة المكرمة. كانت تُسمى يثرب قبل أن يهاجر إليها النبي محمد صلى الله عليه وسلم عام 622 للميلاد، وهو الحدث الذي يبدأ به التقويم الهجري. فيها المسجد النبوي الذي يضم قبر النبي، ومسجد قباء وهو أول مسجد بُني في الإسلام. ويقع شمالها جبل أحد الذي وقعت عنده غزوة أحد في السنة الثالثة للهجرة. تشتهر المدينة بالتمور وخاصة تمر العجوة، وفيها مجمع الملك فهد لطباعة المصحف الشريف.

جدة مدينة ساحلية على البحر الأحمر، وتُعرف ببوابة الحرمين الشريفين لأن كثيرًا من الحجاج والمعتمرين يصلون عبر مطارها ومينائها. فيها ميناء جدة الإسلامي، وهو الميناء الرئيسي للمملكة على البحر الأحمر، ومطار الملك عبدالعزيز الدولي. سُجّلت منطقة جدة التاريخية، المعروفة بالبلد، في قائمة التراث العالمي لليونسكو عام 2014، وتتميز ببيوتها القديمة ذات الرواشين الخشبية. ومن معالمها نافورة الملك فهد، وهي أعلى نافورة في العالم. وتستضيف جدة سباق الفورمولا 1 على حلبة الكورنيش منذ عام 2021.

الدمام عاصمة المنطقة الشرقية، وتقع على ساحل الخليج العربي، وتشكّل مع الخبر والظهران تجمعًا حضريًا واحدًا. ارتبط اسمها بالنفط، ففي عام 1938 اكتُشف النفط بكميات تجارية في بئر الدمام رقم 7 المعروفة ببئر الخير، وتتخذ شركة أرامكو السعودية من الظهران مقرًّا لها. في الدمام ميناء الملك عبدالعزيز، وهو من أكبر موانئ المملكة على الخليج العربي، ويخدمها مطار الملك فهد الدولي، وهو أكبر مطار في العالم من حيث المساحة. ويربط جسر الملك فهد، الذي افتُتح عام 1986، مدينة الخبر بمملكة البحرين.

الطائف مدينة جبلية في منطقة مكة المكرمة، تقع على مرتفعات جبال السروات على ارتفاع يتجاوز 1700 متر فوق سطح البحر. يجعلها مناخها المعتدل مصيفًا يقصده الناس هربًا من حرارة الصيف، ولذلك تُلقَّب بعروس المصايف. تشتهر بالورد الطائفي الذي يُستخرج منه دهن الورد وماء الورد، وبفواكهها مثل الرمان والعنب. ومن أبرز مواقعها منطقتا الهدا والشفا الجبليتان، وسوق عكاظ الذي كان أشهر أسواق العرب قبل الإسلام وكان الشعراء يتبارون فيه.

أبها عاصمة منطقة عسير في جنوب غرب المملكة، وتقع على ارتفاع نحو 2200 متر فوق سطح البحر، ولذلك يبقى طقسها معتدلًا في الصيف وتكثر فيها الأمطار والضباب. بالقرب منها جبل السودة، وهو من أعلى القمم في المملكة ويبلغ ارتفاعه نحو ثلاثة آلاف متر. ومن المواقع التراثية القريبة منها قرية رجال ألمع بمبانيها الحجرية العالية. وتشتهر المنطقة بفن القط العسيري، وهو زخارف ملونة ترسمها النساء على جدران البيوت، وقد سُجّل في قائمة التراث الثقافي غير المادي لليونسكو عام 2017.

تبوك عاصمة منطقة تبوك في شمال غرب المملكة. عُرفت في التاريخ الإسلامي بغزوة تبوك التي وقعت في السنة التاسعة للهجرة، ومن معالمها التاريخية قلعة تبوك ومحطة سكة حديد الحجاز. تعتمد المنطقة على الزراعة وتشتهر بإنتاج الورود والفواكه والقمح. في منطقة تبوك جبل اللوز الذي تتساقط عليه الثلوج في بعض فصول الشتاء، وفيها كذلك مشروع نيوم على ساحل البحر الأحمر، وهو من أكبر المشاريع في رؤية المملكة 2030.

حائل عاصمة منطقة حائل في شمال المملكة، وتقع بين سلسلتي جبال أجا وسلمى. اشتهرت بأنها موطن حاتم الطائي الذي يُضرب به المثل في الكرم عند العرب. من معالمها قلعة أعيرف المبنية على تل يطل على المدينة. وفي منطقة حائل موقعا جبة والشويمس اللذان يضمان نقوشًا ورسومًا صخرية عمرها آلاف السنين، وقد سُجّلا في قائمة التراث العالمي لليونسكو عام 2015. وتستضيف حائل كل عام رالي حائل الدولي في صحراء النفود الكبير.

بريدة عاصمة منطقة القصيم في وسط المملكة، وهي من أهم المدن الزراعية فيها. تشتهر بإنتاج التمور، وخاصة التمر السكري، ويقام فيها كل عام مهرجان للتمور يُعد من أكبر أسواق التمور في العالم. وفيها كذلك سوق للإبل من أكبر أسواق الإبل في العالم. تعتمد الزراعة في القصيم على المياه الجوفية، وتنتج المنطقة إلى جانب التمور القمح والخضراوات. وتقع بالقرب منها مدينة عنيزة، ثاني أكبر مدن القصيم.

الأحساء محافظة في المنطقة الشرقية، وأكبر مدنها الهفوف والمبرز. تضم واحة الأحساء، وهي أكبر واحة نخيل في العالم وفيها نحو مليونين ونصف المليون نخلة، وقد سُجّلت في قائمة التراث العالمي لليونسكو عام 2018. تشتهر الأحساء بتمر الخلاص وبعيون الماء الطبيعية. ومن معالمها جبل القارة المعروف بكهوفه الباردة صيفًا، وقصر إبراهيم التاريخي في الهفوف، ومسجد جواثا وهو من أقدم المساجد في الإسلام.

العلا محافظة في منطقة المدينة المنورة في شمال غرب المملكة، وتشتهر بآثارها وتكويناتها الصخرية. فيها موقع الحِجر، المعروف بمدائن صالح، وهو أول موقع سعودي يُسجَّل في قائمة التراث العالمي لليونسكو، وكان ذلك عام 2008. يضم الحِجر أكثر من مئة مقبرة نحتها الأنباط في الصخر قبل نحو ألفي عام. ومن معالم العلا جبل الفيل، وهو صخرة طبيعية تشبه شكل الفيل، والبلدة القديمة، وقاعة مرايا وهي أكبر مبنى مغطى بالمرايا في العالم.

## Step 1: Load the text file

Read the whole file into one string and count its paragraphs to confirm it has at least 10.

In [ ]:
with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

paragraphs = [p for p in text.split("\n\n") if p.strip()]
print("Characters:", len(text))
print("Paragraphs:", len(paragraphs))

## Step 2: Fixed-size chunking with overlap

A window of 500 characters slides over the text and moves 400 characters each time. The last 100 characters of every chunk are repeated at the start of the next one, so a sentence that is cut at a chunk boundary still appears whole in one of the two chunks.

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100   # 20% of the chunk size

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    step = size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + size]
        if chunk.strip():
            chunks.append(chunk)
        if start + size >= len(text):
            break
    return chunks

chunks = chunk_text(text)
print("Number of chunks:", len(chunks))
print("Overlap between chunk 0 and chunk 1 is identical:", chunks[0][-CHUNK_OVERLAP:] == chunks[1][:CHUNK_OVERLAP])

## Step 3: Embed the chunks

Send the chunks to the embedding model. Each chunk comes back as one vector, and the vectors are stored as rows of a NumPy array. Row `i` belongs to `chunks[i]`.

In [ ]:
def embed(texts, task_type):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    )
    if len(result.embeddings) != len(texts):
        raise RuntimeError(f"Sent {len(texts)} texts but received {len(result.embeddings)} vectors.")
    return np.array([e.values for e in result.embeddings])

chunk_vectors = embed(chunks, "RETRIEVAL_DOCUMENT")
print("Chunk vectors shape:", chunk_vectors.shape)

## Step 4: Get the user's query

The query is typed by the user while the program runs. Example: ما المدينة التي تشتهر بالورد؟

In [ ]:
question = input("Enter your question: ").strip()

## Step 5: Embed the query with the same model

The query goes through the same `embed` function and the same model, so its vector can be compared with the chunk vectors.

In [ ]:
query_vector = embed([question], "RETRIEVAL_QUERY")[0]
print("Query vector shape:", query_vector.shape)

## Step 6: Cosine similarity

Cosine similarity is the dot product of two vectors divided by the product of their lengths. It is computed between the query vector and every chunk vector, which gives one score per chunk.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = [cosine_similarity(query_vector, v) for v in chunk_vectors]
print("Number of scores:", len(scores))

## Step 7: Rank and print the top 3

Sort the scores from highest to lowest, take the first three, and print the question followed by the three chunks with their scores.

In [ ]:
top3 = np.argsort(scores)[::-1][:3]

print("Question:", question)
print()
for rank, i in enumerate(top3, start=1):
    print(f"Answer {rank} (chunk {i}, score = {scores[i]:.4f})")
    print(chunks[i])
    print("-" * 60)